In [35]:
import h5py
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import MinMaxScaler

In [36]:
with h5py.File('../../data/radioml/GOLD_XYZ_OSC.0001_1024.hdf5', 'r') as f:
    print(list(f.keys()))
    print(f['X'].shape)
    print(f['Y'].shape)
    print(f['Z'].shape)

['X', 'Y', 'Z']
(2555904, 1024, 2)
(2555904, 24)
(2555904, 1)


X: 2555904 frames in total, with 4096 frames per unique modulation and unique SNR, resulting in 1024 complex samples per frame

Y: 24 modulations (one-hot encoded) per frame

Z: 4096 frames per unique SNR per unique modulation

In [37]:
with h5py.File('../../data/radioml/GOLD_XYZ_OSC.0001_1024.hdf5', 'r') as f:
    X = f['X'][:50000]    # IQ samples
    Y = f['Y'][:50000]    # one-hot labels
    Z = f['Z'][:50000]    # SNR values

# Convert one-hot → class index (see cell below)
labels = np.argmax(Y, axis=1)   # (50000,)  values 0-23

print(X.shape)        # (50000, 1024, 2)
print(labels.shape)   # (50000,)
print(Z.min(), Z.max())  # SNR range --> -20 to 4

(50000, 1024, 2)
(50000,)
-20 4


an example of a one-hot encoded label: OOK ---> [1, 0, 0, 0, ... 0] (24 numbers)

after converting to label encoded:
- OOK  --> 0
- 4ASK --> 1
- BPSK --> 2
- QPSK --> 3
- ...


- SNR is bad, for initial testing of pipeline, we train using better SNR
- data is formatting in a way where the poorest SNR comes first, and subsequent data has +2 SNR

In [38]:
with h5py.File('../../data/radioml/GOLD_XYZ_OSC.0001_1024.hdf5', 'r') as f:
    # skip first 1 million samples, load next 50000
    X = f['X'][1000000:1050000]
    Y = f['Y'][1000000:1050000]
    Z = f['Z'][1000000:1050000]

labels = np.argmax(Y, axis=1)
snr = Z.flatten()

print(np.unique(snr))   # check SNR range
print(X.shape)

[ 0  2  4  6  8 10 12 14 16 18 20 22 24]
(50000, 1024, 2)


SNR available: -20,-18,-16,-14,-12,-10,-8,-6,-2,0,2,4,6,8,10,12,14,16,18,20,22,24,26,28,30
- if we want SNR>=10, we need to multiply max_frames_per_snr == 4096 by 15

In [39]:
max_frames_per_snr = 4096
snr_above_10 = 15

# Modulation type 0, SNR 10dB Block
start = 4096 * 15
end = start + max_frames_per_snr

with h5py.File('../../data/radioml/GOLD_XYZ_OSC.0001_1024.hdf5', 'r') as f:
    # skip first 1 million samples, load next 50000
    X = f['X'][start:end]
    Y = f['Y'][start:end]
    Z = f['Z'][start:end]

labels = np.argmax(Y, axis=1)
snr = Z.flatten()

print(np.unique(snr))   # check SNR range
print(X.shape)

[10]
(4096, 1024, 2)


In [40]:
frames_per_block = 4096
snr_index = 15   # SNR 10dB
total_modulation = 24
total_snr = 26

X_list, Y_list, Z_list = [], [], []

with h5py.File('../../data/radioml/GOLD_XYZ_OSC.0001_1024.hdf5', 'r') as f:
    for modulation_type in range(total_modulation):
        block_start = (modulation_type * total_snr + snr_index) * frames_per_block
        block_end = block_start + frames_per_block
        X_list.append(f['X'][block_start:block_end])
        Y_list.append(f['Y'][block_start:block_end])
        Z_list.append(f['Z'][block_start:block_end])

X = np.concatenate(X_list)
Y = np.concatenate(Y_list)
Z = np.concatenate(Z_list)

labels = np.argmax(Y, axis=1)

print(X.shape)                 # (98304, 1024, 2)
print(np.unique(labels))       # [0 1 2 ... 23]
print(np.unique(Z.flatten()))  # [10]

(98304, 1024, 2)
[ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21 22 23]
[10]


- 98304 recordings in total, 1024 pairs of IQ samples, 24 modulations, 4096 frames
- all modulation classes are inside
- this is all the IQ samples for ONLY SNR = 10

In [41]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X, labels,
    test_size=0.30,
    random_state=42,
    stratify=labels
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.50,
    random_state=42,
    stratify=y_temp
)

print(f'Train Dataset: {X_train.shape}')    # (68812, 1024, 2)
print(f'Val Dataset: {X_val.shape}')        # (14745, 1024, 2)
print(f'Test Dataset: {X_test.shape}')      # (14747, 1024, 2)

Train Dataset: (68812, 1024, 2)
Val Dataset: (14746, 1024, 2)
Test Dataset: (14746, 1024, 2)


In [42]:
from torch.utils.data import TensorDataset, DataLoader

# Convert to tensors
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
X_val_tensor   = torch.tensor(X_val,   dtype=torch.float32)
X_test_tensor  = torch.tensor(X_test,  dtype=torch.float32)

y_train_tensor = torch.tensor(y_train, dtype=torch.long)
y_val_tensor   = torch.tensor(y_val,   dtype=torch.long)
y_test_tensor  = torch.tensor(y_test,  dtype=torch.long)

# Wrap in datasets
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
val_dataset   = TensorDataset(X_val_tensor,   y_val_tensor)
test_dataset  = TensorDataset(X_test_tensor,  y_test_tensor)

# Create loaders
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=64, shuffle=False)
test_loader  = DataLoader(test_dataset,  batch_size=64, shuffle=False)

print(f"Train batches: {len(train_loader)}")
print(f"Val batches:   {len(val_loader)}")
print(f"Test batches:  {len(test_loader)}")

Train batches: 1076
Val batches:   231
Test batches:  231


In [43]:
class RadioMLLSTM(nn.Module):
    def __init__(self, input_size=2, hidden_size=128, num_layers=2, num_classes=24):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=input_size,  # I & Q
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=0.2
        )
        self.fc = nn.Linear(hidden_size, num_classes)  # 24 classes not 1

    def forward(self, x):
        # x: (batch, 1024, 2)
        out, _ = self.lstm(x)         # (batch, 1024, hidden_size)
        last = out[:, -1, :]          # (batch, hidden_size)
        return self.fc(last)          # (batch, 24)  ← 24 scores, one per class

model = RadioMLLSTM()
print(model)

RadioMLLSTM(
  (lstm): LSTM(2, 128, num_layers=2, batch_first=True, dropout=0.2)
  (fc): Linear(in_features=128, out_features=24, bias=True)
)


In [45]:
from tqdm import tqdm

device = torch.device('mps')
model = model.to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

for epoch in range(10):
    # --- training ---
    model.train()
    total_loss, correct, total = 0, 0, 0

    train_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/10 [Train]")
    for X_batch, y_batch in train_bar:                                      # Iterates through DataLoader 1 batch at a time
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)           # moves batch to GPU

        optimizer.zero_grad()
        logits = model(X_batch)             # calls forward under the hood, returns raw scores (logits) for each 24 classes
        loss = criterion(logits, y_batch)   # Computes how wrong values were, error
        loss.backward()                     # backpropagation for changing weights
        optimizer.step()                    # change the weights

        total_loss += loss.item()
        correct += (logits.argmax(1) == y_batch).sum().item()
        # logits.argmax(1)
        # pick highest scoring class per example → (64,) == y_batch
        # compare to true label → (64,) of True/False.sum()
        # count how many were correct → one number.item()
        # convert tensor to Python int
        
        total += y_batch.size(0)

        # update progress bar with live stats
        train_bar.set_postfix(loss=f"{loss.item():.4f}", acc=f"{correct/total:.4f}")

    # --- validation ---
    model.eval()
    val_correct, val_total = 0, 0

    val_bar = tqdm(val_loader, desc=f"Epoch {epoch+1}/10 [Val]  ")
    with torch.no_grad():
        for X_batch, y_batch in val_bar:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            logits = model(X_batch)
            val_correct += (logits.argmax(1) == y_batch).sum().item()
            val_total += y_batch.size(0)

            val_bar.set_postfix(acc=f"{val_correct/val_total:.4f}")

    print(f"Epoch {epoch+1} summary — loss={total_loss:.4f}  train_acc={correct/total:.4f}  val_acc={val_correct/val_total:.4f}\n")

Epoch 1/10 [Train]:   0%|          | 0/1076 [00:00<?, ?it/s]

Epoch 1/10 [Val]  : 100%|██████████| 231/231 [00:12<00:00, 18.92it/s, acc=0.2097]


Epoch 1 summary — loss=2726.7942  train_acc=0.1641  val_acc=0.2097



Epoch 2/10 [Val]  : 100%|██████████| 231/231 [00:11<00:00, 19.33it/s, acc=0.3221]


Epoch 2 summary — loss=2237.7641  train_acc=0.2684  val_acc=0.3221



...

Epoch 8 summary — loss=1101.8564  train_acc=0.5666  val_acc=0.5629



Epoch 9/10 [Val]  : 100%|██████████| 231/231 [00:20<00:00, 11.07it/s, acc=0.5496]


Epoch 9 summary — loss=998.0945  train_acc=0.6000  val_acc=0.5496



Epoch 10/10 [Val]  : 100%|██████████| 231/231 [00:21<00:00, 10.57it/s, acc=0.6388]

Epoch 10 summary — loss=918.5038  train_acc=0.6285  val_acc=0.6388

